# Notebook 4: Neural Network (PyTorch)
**Goal:** Fit a feedforward neural network and compare against GLM and XGBoost.  
**Why PyTorch?** Industry standard deep learning framework. We train on log(ClaimAmount) since neural nets work better with normally distributed targets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/claims_cleaned.csv')

features = ['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density', 'Area', 'VehBrand', 'VehGas', 'Region']
target = 'ClaimAmount'

df_model = pd.get_dummies(df[features + [target]], columns=['Area', 'VehBrand', 'VehGas', 'Region'], drop_first=True)

X = df_model.drop(columns=[target]).values.astype(np.float32)
y = np.log1p(df_model[target].values.astype(np.float32))  # log-transform target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features — important for neural nets
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Define Model

In [ ]:
class ClaimNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model = ClaimNet(input_dim=X_train.shape[1]).to(device)
print(model)

## 3. Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()  # MSE on log scale ≈ minimising proportional errors

epochs = 100
train_losses, val_losses = [], []

for epoch in range(epochs):
    # Training
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_losses.append(np.mean(batch_losses))

    # Validation
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test_t), y_test_t).item()
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}')

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (log scale)')
plt.title('Neural Net Training Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../results/06_nn_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_log = model(X_test_t).cpu().numpy().flatten()

# Inverse log transform to get back to EUR
y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

mae  = mean_absolute_error(y_test_orig, y_pred)
rmse = mean_squared_error(y_test_orig, y_pred) ** 0.5

def gamma_deviance(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-6, None)  # avoid division by zero
    return 2 * np.mean((y_true - y_pred) / y_pred - np.log(y_true / y_pred))

gd = gamma_deviance(y_test_orig, y_pred)

print('--- Neural Net Test Set Performance ---')
print(f'MAE:             {mae:,.2f}')
print(f'RMSE:            {rmse:,.2f}')
print(f'Gamma Deviance:  {gd:.4f}')

nn_metrics = {'model': 'NeuralNet', 'MAE': mae, 'RMSE': rmse, 'GammaDeviance': gd}

## 5. Model Comparison

In [ ]:
# TODO: paste in glm_metrics and xgb_metrics from the earlier notebooks, then run this cell
# glm_metrics = {'model': 'GLM', 'MAE': ..., 'RMSE': ..., 'GammaDeviance': ...}
# xgb_metrics = {'model': 'XGBoost', 'MAE': ..., 'RMSE': ..., 'GammaDeviance': ...}

results = pd.DataFrame([glm_metrics, xgb_metrics, nn_metrics])
results = results.set_index('model')
print(results.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'GammaDeviance']):
    results[metric].plot(kind='bar', ax=ax, edgecolor='white')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../results/07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()